# LeetCode 527 — Word Abbreviation bằng Python

- Link: https://leetcode.com/problems/word-abbreviation/
- Độ khó: **Hard** (bài **Premium**)
- Chủ đề: String, Greedy, Sorting, Longest Common Prefix (LCP), Trie

> **Ghi chú Premium:** Đây là bài dành cho tài khoản LeetCode Premium. Phần đề bài dưới đây được viết lại theo hiểu biết phổ biến về bài toán (quy tắc viết tắt, cách giải quyết trùng, ví dụ, ràng buộc). Nếu bạn có tài khoản Premium, hãy **đối chiếu lại** câu chữ, ví dụ và ràng buộc chính xác trên LeetCode.

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def wordsAbbreviation(self, words: List[str]) -> List[str]:

```

## 1. Mục tiêu học

1. Cài đặt chính xác **quy tắc viết tắt**: tiền tố + số ký tự bị lược + ký tự cuối, và quy tắc "không ngắn hơn thì giữ nguyên".
2. **Mô phỏng** đúng luật giải quyết trùng: nhóm từ trùng viết tắt cùng tăng độ dài tiền tố.
3. Rút ra insight: độ dài tiền tố cuối cùng của một từ = **1 + LCP lớn nhất** với các từ "cùng lớp" (cùng độ dài, cùng ký tự đầu, cùng ký tự cuối).
4. Tính LCP lớn nhất nhanh bằng **sắp xếp + so với hàng xóm**, hoặc bằng **Trie có đếm**.
5. Kiểm thử chéo 4 cách với mô phỏng trên dữ liệu ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng `words` gồm các từ **phân biệt**. Hãy trả về **viết tắt ngắn nhất có thể** của mỗi từ theo quy tắc:

1. Viết tắt ban đầu của một từ = **ký tự đầu** + **số ký tự ở giữa** + **ký tự cuối**. Ví dụ `"internal"` → `"i6l"`.
2. Nếu **nhiều từ có cùng viết tắt**, thì với **mọi** từ trong nhóm trùng đó, tăng tiền tố thêm 1 ký tự (ví dụ `"in5l"`), và lặp lại cho tới khi mọi viết tắt là duy nhất.
3. Nếu viết tắt **không ngắn hơn** từ gốc, giữ nguyên từ gốc.

Trả về danh sách viết tắt theo đúng thứ tự `words`.

**Ví dụ 1**

```text
Input:  words = ["like","god","internal","me","internet","interval","intension","face","intrusion"]
Output: ["l2e","god","internal","me","i6t","interval","inte4n","f2e","intr4n"]
```

**Ví dụ 2**

```text
Input:  words = ["aa","aaa"]
Output: ["aa","aaa"]
```

**Ràng buộc (theo hiểu biết, nên đối chiếu)**

- `1 <= words.length <= 400`
- `2 <= words[i].length <= 400`
- `words[i]` chỉ gồm chữ thường tiếng Anh.
- Mọi từ trong `words` **khác nhau**.

In [1]:
def abbr(word, p):
    """Viết tắt với tiền tố dài p; nếu không ngắn hơn thì trả về từ gốc."""
    L = len(word)
    if L - p - 1 <= 1:            # còn <= 1 ký tự ở giữa -> viết tắt không ngắn hơn
        return word
    s = word[:p] + str(L - p - 1) + word[-1]
    return s if len(s) < L else word

for w in ["internal", "like", "god", "me", "intension"]:
    print(w, "->", [abbr(w, p) for p in range(1, 5)])

internal -> ['i6l', 'in5l', 'int4l', 'inte3l']
like -> ['l2e', 'like', 'like', 'like']
god -> ['god', 'god', 'god', 'god']
me -> ['me', 'me', 'me', 'me']
intension -> ['i7n', 'in6n', 'int5n', 'inte4n']


## 3. Kiến thức nền

### 3.1. Slicing chuỗi: `word[:p]`, `word[-1]`, `len`

- `word[:p]`: `p` ký tự đầu.
- `word[-1]`: ký tự cuối.
- Số ký tự bị lược = `len(word) - p - 1`.

In [2]:
w = "intension"
p = 4
print(w[:p], len(w) - p - 1, w[-1], "->", w[:p] + str(len(w) - p - 1) + w[-1])

inte 4 n -> inte4n


### 3.2. Khi nào viết tắt **không ngắn hơn**?

Viết tắt dài `p + (số chữ số của k) + 1` với `k = L - p - 1`. Nó ngắn hơn từ gốc (`L = p + k + 1`) khi và chỉ khi `số chữ số của k < k`, tức `k >= 2`. Vậy chỉ cần kiểm tra `k <= 1` → giữ nguyên. (Ví dụ `"god"`: `k = 1` → `"g1d"` dài bằng `"god"` → giữ `"god"`.)

In [3]:
for k in range(0, 13):
    print(k, len(str(k)) < k, end=" | ")

0 False | 1 False | 2 True | 3 True | 4 True | 5 True | 6 True | 7 True | 8 True | 9 True | 10 True | 11 True | 12 True | 

### 3.3. Đếm và nhóm bằng `Counter` / `defaultdict(list)`

Để phát hiện viết tắt trùng: đếm số lần mỗi viết tắt xuất hiện; viết tắt nào xuất hiện ≥ 2 lần là trùng.

In [4]:
from collections import Counter, defaultdict
abbrs = ["i6t", "i6l", "i6l", "l2e"]
cnt = Counter(abbrs)
print(cnt, "| trùng:", [a for a, c in cnt.items() if c > 1])
groups = defaultdict(list)
for i, a in enumerate(abbrs):
    groups[a].append(i)
print(dict(groups))

Counter({'i6l': 2, 'i6t': 1, 'l2e': 1}) | trùng: ['i6l']
{'i6t': [0], 'i6l': [1, 2], 'l2e': [3]}


### 3.4. Longest Common Prefix (LCP) của hai chuỗi

`lcp(a, b)` = số ký tự đầu giống nhau. Duyệt song song bằng `zip`.

In [5]:
def lcp(a, b):
    k = 0
    for x, y in zip(a, b):
        if x != y:
            break
        k += 1
    return k

print(lcp("internal", "interval"), lcp("intension", "intrusion"), lcp("abc", "xyz"))

5 3 0


### 3.5. Tính chất của danh sách đã sắp xếp: LCP lớn nhất nằm ở hàng xóm

Sắp xếp các chuỗi theo thứ tự từ điển. Với một chuỗi `s`, **LCP lớn nhất** giữa `s` và mọi chuỗi khác đạt được ở **hàng xóm liền trước hoặc liền sau** trong danh sách đã sắp. Lý do: các chuỗi cùng tiền tố dài `t` với `s` nằm thành một **khối liên tiếp** chứa `s`.

In [6]:
ws = sorted(["internal", "interval", "internet", "intension", "intrusion"])
print(ws)
for i, s in enumerate(ws):
    best_any = max(lcp(s, t) for t in ws if t != s)
    nb = max([lcp(s, ws[j]) for j in (i - 1, i + 1) if 0 <= j < len(ws)])
    print(f"{s:<10} LCP max với mọi từ = {best_any}, với hàng xóm = {nb}")

['intension', 'internal', 'internet', 'interval', 'intrusion']
intension  LCP max với mọi từ = 4, với hàng xóm = 4
internal   LCP max với mọi từ = 6, với hàng xóm = 6
internet   LCP max với mọi từ = 6, với hàng xóm = 6
interval   LCP max với mọi từ = 5, với hàng xóm = 5
intrusion  LCP max với mọi từ = 3, với hàng xóm = 3


### 3.6. Trie có đếm

**Trie** (cây tiền tố): mỗi nút ứng với một tiền tố. Lưu `count` = số từ đi qua nút. Với một từ, đi từ gốc xuống; nút đầu tiên có `count == 1` cho biết tiền tố tới đó đã **riêng** của từ này.

In [7]:
def build_trie(words):
    root = {}
    for w in words:
        node = root
        for ch in w:
            node = node.setdefault(ch, {"#": 0})
            node["#"] += 1
    return root

t = build_trie(["internal", "interval"])
node = t
for ch in "internal":
    node = node[ch]
    print(ch, "count =", node["#"])

i count = 2
n count = 2
t count = 2
e count = 2
r count = 2
n count = 1
a count = 1
l count = 1


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1. Viết tắt ban đầu (`p = 1`):

| từ | viết tắt p=1 |
|---|---|
| like | l2e |
| god | god (giữ nguyên) |
| internal | i6l |
| me | me |
| internet | i6t |
| interval | i6l |
| intension | i7n |
| face | f2e |
| intrusion | i7n |

Trùng: `i6l` (internal, interval), `i7n` (intension, intrusion).
- internal/interval: tăng dần `p = 2 "in5l"`, `3 "int4l"`, `4 "inte3l"`, `5 "inter2l"` vẫn trùng; `p = 6` → `"intern1l"` / `"interv1l"` — **không ngắn hơn** → giữ nguyên `"internal"`, `"interval"`.
- intension/intrusion: `p = 2 "in6n"`, `3 "int5n"` trùng; `p = 4`: `"inte4n"` vs `"intr4n"` → khác nhau. ✔

Nhận xét: `internal` và `interval` có LCP = 5 (`"inter"`), nên cần `p = 6`. `intension`/`intrusion` có LCP = 3 (`"int"`), cần `p = 4`. Gợi ý mạnh: **`p = 1 + LCP`**.

In [8]:
words = ["like", "god", "internal", "me", "internet", "interval", "intension", "face", "intrusion"]
print([abbr(w, 1) for w in words])

['l2e', 'god', 'i6l', 'me', 'i6t', 'i6l', 'i7n', 'f2e', 'i7n']


## 5. Cách 1 — Brute Force: mô phỏng đúng luật

**Ý tưởng.** `pre[i] = 1` cho mọi từ. Lặp:
1. Tính `ab[i] = abbr(words[i], pre[i])`.
2. Đếm; với mọi `i` có `ab[i]` bị trùng → `pre[i] += 1`.
3. Nếu không còn trùng → dừng.

In [9]:
def abbrev_simulate(words):
    n = len(words)
    pre = [1] * n
    ab = [abbr(w, 1) for w in words]
    while True:
        cnt = Counter(ab)
        dup = [i for i in range(n) if cnt[ab[i]] > 1]
        if not dup:
            return ab
        for i in dup:
            pre[i] += 1
            ab[i] = abbr(words[i], pre[i])

ex1 = ["like", "god", "internal", "me", "internet", "interval", "intension", "face", "intrusion"]
out1 = ["l2e", "god", "internal", "me", "i6t", "interval", "inte4n", "f2e", "intr4n"]
print(abbrev_simulate(ex1))
assert abbrev_simulate(ex1) == out1
assert abbrev_simulate(["aa", "aaa"]) == ["aa", "aaa"]

['l2e', 'god', 'internal', 'me', 'i6t', 'interval', 'inte4n', 'f2e', 'intr4n']


Lưu ý quan trọng trong mô phỏng: ta tính `dup` **trước**, rồi mới tăng tiền tố cho tất cả — tránh việc tăng một từ làm "biến mất" trùng lặp của từ kia giữa chừng (luật nói **mọi** từ trong nhóm trùng đều tăng).

**Độ phức tạp:** mỗi vòng `O(n · L)` (tạo viết tắt + đếm); số vòng tối đa `L` → Time `O(n · L²)` = `400 · 400² = 6.4·10^7` thao tác ký tự — chậm trong Python ở trường hợp xấu. Space `O(n · L)`.

**Nút thắt:** mỗi vòng ta dựng lại viết tắt của mọi từ trùng, và từ có tiền tố chung dài phải đi qua rất nhiều vòng.

In [10]:
import time
words_bad = ["a" * 200 + chr(97 + i) + "z" for i in range(26)] + ["b" * 200 + chr(97 + i) + "z" for i in range(26)]
t = time.perf_counter()
abbrev_simulate(words_bad)
print(f"mô phỏng với 52 từ dài 202, tiền tố chung 200: {time.perf_counter() - t:.3f}s")

mô phỏng với 52 từ dài 202, tiền tố chung 200: 0.005s


## 6. Bước đột phá

**Insight 1 — Chỉ các từ "cùng lớp" mới có thể trùng.** Hai viết tắt bằng nhau ⇒ cùng tiền tố (ký tự chữ, không chứa số), cùng con số (⇒ cùng độ dài), cùng ký tự cuối. Vì từ chỉ có chữ cái, cách đọc "phần chữ / phần số / chữ cuối" là duy nhất. Do đó định nghĩa **lớp** của từ là `(len, ký tự đầu, ký tự cuối)`; từ khác lớp không bao giờ trùng.

**Insight 2 — Độ dài tiền tố cuối cùng = 1 + LCP lớn nhất trong lớp.** Theo luật, các từ trùng nhau ở bước `p` có cùng tiền tố dài `p` và cùng tăng lên `p + 1`. Hai từ "tách nhau" ngay khi tiền tố khác nhau, và sau đó không bao giờ trùng lại (tiền tố dài hơn cũng khác). Vậy từ `w` dừng ở `p` nhỏ nhất sao cho **không** từ nào khác trong lớp có cùng tiền tố dài `p`, tức

```text
p(w) = 1 + max_{u cùng lớp, u ≠ w} lcp(w, u)      (nếu lớp chỉ có w thì p = 1)
```

rồi áp dụng `abbr(w, p)` (tự động giữ nguyên nếu không ngắn hơn).

**Insight 3 — LCP lớn nhất tìm bằng hàng xóm sau khi sắp xếp** (mục 3.5), hoặc bằng Trie đếm (mục 3.6).

Quy tắc: **nếu** từ là duy nhất trong lớp **thì** `p = 1`; **ngược lại** `p = 1 + max(lcp với hàng xóm trái, lcp với hàng xóm phải)`.

In [11]:
# Minh họa Insight 2 trên lớp (len=8, 'i', 'l'): internal, interval
for w, u in [("internal", "interval"), ("intension", "intrusion")]:
    p = 1 + lcp(w, u)
    print(w, u, "LCP =", lcp(w, u), "-> p =", p, "->", abbr(w, p), abbr(u, p))

internal interval LCP = 5 -> p = 6 -> internal interval
intension intrusion LCP = 3 -> p = 4 -> inte4n intr4n


## 7. Các cách tiếp theo

### 7.1. Cách 2 — LCP theo cặp trong từng lớp (`O(n² L)`)

Với mỗi từ, so với mọi từ cùng lớp để tìm LCP lớn nhất.

In [12]:
def abbrev_pairwise(words):
    groups = defaultdict(list)
    for i, w in enumerate(words):
        groups[(len(w), w[0], w[-1])].append(i)
    res = [None] * len(words)
    for idxs in groups.values():
        for i in idxs:
            best = 0
            for j in idxs:
                if j != i:
                    best = max(best, lcp(words[i], words[j]))
            res[i] = abbr(words[i], best + 1)
    return res

print(abbrev_pairwise(ex1))
assert abbrev_pairwise(ex1) == out1

['l2e', 'god', 'internal', 'me', 'i6t', 'interval', 'inte4n', 'f2e', 'intr4n']


**Code nộp LeetCode (Cách 2):**

In [13]:
from typing import List

class SolutionPairwise:
    def wordsAbbreviation(self, words: List[str]) -> List[str]:
        return abbrev_pairwise(words)

print(SolutionPairwise().wordsAbbreviation(["aa", "aaa"]))

['aa', 'aaa']


**Độ phức tạp Cách 2:** xấu nhất (mọi từ cùng lớp) `O(n² · L)` = `400² · 400 = 6.4·10^7` — nhưng mỗi phép so sánh LCP dừng sớm khi gặp ký tự khác, thực tế nhanh. Space `O(n)`.

### 7.2. Cách 3 — Sắp xếp trong lớp + LCP với hàng xóm (lời giải chính)

1. Gom từ theo lớp `(len, đầu, cuối)` (lưu chỉ số).
2. Trong mỗi lớp, sắp xếp chỉ số theo từ.
3. Với mỗi vị trí `k` trong danh sách đã sắp: `best = max(lcp với k-1, lcp với k+1)`; `p = best + 1`.

In [14]:
def abbrev_sorted(words):
    groups = defaultdict(list)
    for i, w in enumerate(words):
        groups[(len(w), w[0], w[-1])].append(i)
    res = [None] * len(words)
    for idxs in groups.values():
        idxs.sort(key=lambda i: words[i])
        m = len(idxs)
        lcps = [lcp(words[idxs[k]], words[idxs[k + 1]]) for k in range(m - 1)]   # LCP các cặp kề
        for k, i in enumerate(idxs):
            best = max(lcps[k - 1] if k > 0 else 0, lcps[k] if k < m - 1 else 0)
            res[i] = abbr(words[i], best + 1)
    return res

print(abbrev_sorted(ex1))
assert abbrev_sorted(ex1) == out1

['l2e', 'god', 'internal', 'me', 'i6t', 'interval', 'inte4n', 'f2e', 'intr4n']


**Code nộp LeetCode (Cách 3):** xem mục 9.

**Độ phức tạp Cách 3:** sắp xếp `O(n log n · L)` (so sánh chuỗi), LCP kề `O(n · L)` → Time `O(n L log n)`. Space `O(n)` (không tính đầu ra).

### 7.3. Cách 4 — Trie có đếm cho từng lớp

Trong mỗi lớp, dựng trie đếm. Với mỗi từ, đi xuống từng ký tự; vị trí `d` (1-based) đầu tiên mà nút có `count == 1` là `p` (tiền tố dài `d` đã riêng). Nếu không gặp (tiền tố trùng tới hết — không xảy ra vì các từ phân biệt cùng độ dài), lấy `p = len(word)`.

In [15]:
def abbrev_trie(words):
    groups = defaultdict(list)
    for i, w in enumerate(words):
        groups[(len(w), w[0], w[-1])].append(i)
    res = [None] * len(words)
    for idxs in groups.values():
        root = build_trie([words[i] for i in idxs])
        for i in idxs:
            w = words[i]
            node, p = root, len(w)
            for d, ch in enumerate(w, 1):
                node = node[ch]
                if node["#"] == 1:
                    p = d
                    break
            res[i] = abbr(w, p)
    return res

print(abbrev_trie(ex1))
assert abbrev_trie(ex1) == out1

['l2e', 'god', 'internal', 'me', 'i6t', 'interval', 'inte4n', 'f2e', 'intr4n']


**Code nộp LeetCode (Cách 4):**

In [16]:
class SolutionTrie:
    def wordsAbbreviation(self, words: List[str]) -> List[str]:
        return abbrev_trie(words)

print(SolutionTrie().wordsAbbreviation(ex1))

['l2e', 'god', 'internal', 'me', 'i6t', 'interval', 'inte4n', 'f2e', 'intr4n']


**Độ phức tạp Cách 4:** dựng trie `O(n · L)`, mỗi từ đi xuống tối đa `L` → Time `O(n · L)`, Space `O(n · L)` cho trie.

Chú ý: với lớp chỉ có một từ, nút đầu tiên (`d = 1`) đã có `count == 1` → `p = 1` — đúng như mong đợi.

## 8. TRACE từng bước

### 8.1. Trace Cách 1 (mô phỏng) trên ví dụ 1

Mỗi vòng in: viết tắt hiện tại của các từ còn bị trùng và độ dài tiền tố.

In [17]:
def trace_simulate(words):
    n = len(words)
    pre = [1] * n
    ab = [abbr(w, 1) for w in words]
    rnd = 0
    while True:
        cnt = Counter(ab)
        dup = [i for i in range(n) if cnt[ab[i]] > 1]
        print(f"vòng {rnd}: " + ", ".join(f"{words[i]}->{ab[i]}(p={pre[i]})" for i in dup) if dup else f"vòng {rnd}: không còn trùng")
        if not dup:
            return ab
        for i in dup:
            pre[i] += 1
            ab[i] = abbr(words[i], pre[i])
        rnd += 1

print(trace_simulate(ex1))

vòng 0: internal->i6l(p=1), interval->i6l(p=1), intension->i7n(p=1), intrusion->i7n(p=1)
vòng 1: internal->in5l(p=2), interval->in5l(p=2), intension->in6n(p=2), intrusion->in6n(p=2)
vòng 2: internal->int4l(p=3), interval->int4l(p=3), intension->int5n(p=3), intrusion->int5n(p=3)
vòng 3: internal->inte3l(p=4), interval->inte3l(p=4)
vòng 4: internal->inter2l(p=5), interval->inter2l(p=5)
vòng 5: không còn trùng
['l2e', 'god', 'internal', 'me', 'i6t', 'interval', 'inte4n', 'f2e', 'intr4n']


### 8.2. Trace Cách 3 trên lớp `(len=9, 'i', 'n')` và một lớp lớn hơn

Bảng cho lớp gồm `intension`, `intrusion`, `inversion` (tự thêm để minh họa 3 từ):

| sắp xếp | LCP với trước | LCP với sau | p | viết tắt |
|---|---|---|---|---|
| intension | – | 3 | 4 | inte4n |
| intrusion | 3 | 2 | 4 | intr4n |
| inversion | 2 | – | 3 | inv5n |

In [18]:
def trace_sorted(words):
    groups = defaultdict(list)
    for i, w in enumerate(words):
        groups[(len(w), w[0], w[-1])].append(i)
    for key, idxs in groups.items():
        idxs.sort(key=lambda i: words[i])
        m = len(idxs)
        lcps = [lcp(words[idxs[k]], words[idxs[k + 1]]) for k in range(m - 1)]
        print(f"lớp {key}: {[words[i] for i in idxs]}  LCP kề = {lcps}")
        for k, i in enumerate(idxs):
            left = lcps[k - 1] if k > 0 else None
            right = lcps[k] if k < m - 1 else None
            best = max(left or 0, right or 0)
            print(f"    {words[i]:<10} trái={left} phải={right} -> p={best + 1} -> {abbr(words[i], best + 1)}")

trace_sorted(["intension", "intrusion", "inversion", "like", "lake", "internal", "interval"])

lớp (9, 'i', 'n'): ['intension', 'intrusion', 'inversion']  LCP kề = [3, 2]
    intension  trái=None phải=3 -> p=4 -> inte4n
    intrusion  trái=3 phải=2 -> p=4 -> intr4n
    inversion  trái=2 phải=None -> p=3 -> inv5n
lớp (4, 'l', 'e'): ['lake', 'like']  LCP kề = [1]
    lake       trái=None phải=1 -> p=2 -> lake
    like       trái=1 phải=None -> p=2 -> like
lớp (8, 'i', 'l'): ['internal', 'interval']  LCP kề = [5]
    internal   trái=None phải=5 -> p=6 -> internal
    interval   trái=5 phải=None -> p=6 -> interval


## 9. Code nộp LeetCode

In [19]:
from collections import defaultdict
from typing import List

class Solution:
    def wordsAbbreviation(self, words: List[str]) -> List[str]:
        def abbr(w: str, p: int) -> str:
            k = len(w) - p - 1                    # số ký tự bị lược
            return w if k <= 1 else w[:p] + str(k) + w[-1]

        def lcp(a: str, b: str) -> int:
            k = 0
            while k < len(a) and a[k] == b[k]:   # cùng lớp nên cùng độ dài
                k += 1
            return k

        groups = defaultdict(list)                # (độ dài, đầu, cuối) -> chỉ số
        for i, w in enumerate(words):
            groups[(len(w), w[0], w[-1])].append(i)

        res = [""] * len(words)
        for idxs in groups.values():
            idxs.sort(key=lambda i: words[i])
            m = len(idxs)
            adj = [lcp(words[idxs[k]], words[idxs[k + 1]]) for k in range(m - 1)]
            for k, i in enumerate(idxs):
                best = max(adj[k - 1] if k > 0 else 0, adj[k] if k < m - 1 else 0)
                res[i] = abbr(words[i], best + 1)
        return res

sol = Solution()
assert sol.wordsAbbreviation(ex1) == out1
assert sol.wordsAbbreviation(["aa", "aaa"]) == ["aa", "aaa"]
print("Solution OK")

Solution OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (đọc viết tắt duy nhất).** Viết tắt có dạng `chữ…chữ` + `số` + `chữ`. Vì từ chỉ gồm chữ cái, từ một viết tắt ta đọc ra duy nhất: tiền tố (phần chữ đầu), `k` (phần số), chữ cuối; suy ra độ dài `p + k + 1`. Viết tắt "giữ nguyên" không chứa chữ số nên không thể bằng viết tắt có số, và hai từ giữ nguyên thì khác nhau (từ phân biệt). ⇒ Hai từ chỉ có thể trùng viết tắt khi cùng lớp `(len, đầu, cuối)`, cùng `p`, cùng tiền tố dài `p`.

**Bổ đề 2 (động học của luật trùng).** Trong mô phỏng, xét hai từ `w, u` cùng lớp với `t = lcp(w, u)`. Khi cả hai cùng `p <= t` thì chúng trùng (nếu viết tắt còn ngắn hơn) và cùng tăng `p`. Khi `p > t` thì tiền tố khác nhau ⇒ không trùng, và mãi mãi không trùng nữa. Bằng quy nạp, mọi từ trong một nhóm trùng luôn có cùng `p` (bắt đầu đều là 1, cùng tăng). Do đó `w` dừng tại `p = 1 + max lcp(w, u)` (hoặc sớm hơn nếu viết tắt đã "giữ nguyên" — kết quả vẫn là từ gốc, và `abbr(w, 1 + max lcp)` cũng trả về từ gốc vì `p` càng lớn càng không ngắn hơn).

**Bổ đề 3 (hàng xóm).** Trong danh sách đã sắp, với `a < b < c` thì `lcp(a, c) <= min(lcp(a, b), lcp(b, c))`. Vì vậy với từ ở vị trí `k`, LCP với bất kỳ từ xa hơn không vượt quá LCP với hàng xóm gần phía đó.

Kết hợp ba bổ đề: Cách 3 tính đúng `p(w)` cho mọi từ, nên kết quả trùng với mô phỏng đúng luật.

## 11. Phân tích Time/Space

Gọi `n` = số từ, `L` = độ dài tối đa.

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng | `O(n · L²)` | `O(n · L)` |
| 2. LCP theo cặp | `O(n² · L)` | `O(n)` |
| 3. Sắp xếp + hàng xóm | `O(n L log n)` | `O(n)` |
| 4. Trie | `O(n · L)` | `O(n · L)` |

In [20]:
import random
random.seed(527)
def random_words(n, L, alpha, common):
    out = set()
    while len(out) < n:
        out.add(common + "".join(random.choice(alpha) for _ in range(L - len(common))))
    return list(out)

big = random_words(400, 400, "ab", "a" * 300)
for name, f in [("simulate", abbrev_simulate), ("pairwise", abbrev_pairwise), ("sorted", abbrev_sorted), ("trie", abbrev_trie), ("Solution", Solution().wordsAbbreviation)]:
    t = time.perf_counter()
    r = f(big)
    print(f"{name:<9}: {time.perf_counter() - t:.3f}s")

simulate : 0.072s


pairwise : 0.591s
sorted   : 0.003s
trie     : 0.020s
Solution : 0.004s


## 12. Bộ test

In [21]:
methods = {
    "simulate": abbrev_simulate,
    "pairwise": abbrev_pairwise,
    "sorted": abbrev_sorted,
    "trie": abbrev_trie,
    "SolPairwise": SolutionPairwise().wordsAbbreviation,
    "SolTrie": SolutionTrie().wordsAbbreviation,
    "Solution": Solution().wordsAbbreviation,
}
cases = [
    (ex1, out1),
    (["aa", "aaa"], ["aa", "aaa"]),
    (["abcdef"], ["a4f"]),                               # một từ
    (["abcd", "abed"], ["abcd", "abed"]),                # LCP = 2 -> p = 3 -> k = 0 -> giữ nguyên
    (["abcdefg", "abcxefg"], ["abcd2g", "abcx2g"]),        # LCP = 3 -> p = 4
    (["apple", "apply", "ample"], ["ap2e", "a3y", "am2e"]),  # apply khác lớp
    (["xyz", "abc"], ["xyz", "abc"]),                    # độ dài 3: luôn giữ nguyên
]
for words, exp in cases:
    for name, f in methods.items():
        got = f(list(words))
        assert got == exp, (name, words, got, exp)
print("Ví dụ + edge case: OK")

Ví dụ + edge case: OK


Giải thích vài edge case:
- `["abcdefg", "abcxefg"]`: cùng lớp `(7, 'a', 'g')`, LCP = 3 → `p = 4` → `"abcd2g"`, `"abcx2g"`.
- `["apple", "apply", "ample"]`: `apply` khác lớp (kết thúc bằng `y`) → `"a3y"`. `apple`/`ample` LCP = 1 → `p = 2` → `"ap2e"`, `"am2e"`.
- `["abcd", "abed"]`: LCP = 2 → `p = 3` → còn 0 ký tự ở giữa → giữ nguyên.

In [22]:
random.seed(2026)
for trial in range(800):
    alpha = random.choice(["ab", "abc"])
    n = random.randint(1, 12)
    words = set()
    while len(words) < n:
        L = random.randint(2, 8)
        words.add("".join(random.choice(alpha) for _ in range(L)))
    words = list(words)
    exp = abbrev_simulate(words)
    assert len(set(exp)) == len(exp)                    # kết quả phải duy nhất
    for name, f in methods.items():
        got = f(list(words))
        assert got == exp, (name, words, got, exp)
print("800 test ngẫu nhiên (seed 2026): mọi cách khớp mô phỏng đúng luật")

800 test ngẫu nhiên (seed 2026): mọi cách khớp mô phỏng đúng luật


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ tăng tiền tố cho **một** từ trong nhóm trùng

**Code sai:** khi phát hiện `ab[i] == ab[j]`, chỉ tăng `pre[j]`.

**Vì sao sai:** luật yêu cầu **mọi** từ trong nhóm trùng cùng tăng. Tăng một từ có thể cho viết tắt ngắn hơn cho từ kia nhưng **sai đề**.

In [23]:
def abbrev_wrong_one(words):
    n = len(words); pre = [1] * n
    ab = [abbr(w, 1) for w in words]
    changed = True
    while changed:
        changed = False
        seen = {}
        for i in range(n):
            if ab[i] in seen:                 # chỉ tăng từ đến sau
                pre[i] += 1; ab[i] = abbr(words[i], pre[i]); changed = True
            else:
                seen[ab[i]] = i
    return ab

w = ["intension", "intrusion"]
print("sai:", abbrev_wrong_one(w), "| đúng:", Solution().wordsAbbreviation(w))

sai: ['i7n', 'in6n'] | đúng: ['inte4n', 'intr4n']


**Code đúng:** thu thập toàn bộ chỉ số trùng trước, rồi tăng cho tất cả (như `abbrev_simulate`).

### Lỗi 2 — Quên quy tắc "không ngắn hơn thì giữ nguyên" (hoặc kiểm tra sai)

**Code sai:** luôn trả `w[:p] + str(len(w)-p-1) + w[-1]`.

In [24]:
def abbr_wrong(w, p):
    return w[:p] + str(len(w) - p - 1) + w[-1]
print("sai:", abbr_wrong("god", 1), abbr_wrong("me", 1), "| đúng:", abbr("god", 1), abbr("me", 1))

sai: g1d m0e | đúng: god me


**Code đúng:** `k = len(w) - p - 1`; nếu `k <= 1` trả `w`. (Lưu ý `"me"` cho `k = 0` → `"m0e"` dài hơn từ gốc!)

### Lỗi 3 — So LCP với mọi từ (không chia lớp)

**Code sai:** tính `p = 1 + max lcp(w, u)` với **mọi** `u` khác, kể cả khác độ dài/ký tự cuối.

**Vì sao sai:** từ khác lớp không bao giờ trùng viết tắt, nhưng LCP với chúng có thể lớn → tiền tố dài thừa, không còn "ngắn nhất".

In [25]:
def abbrev_no_class(words):
    res = []
    for i, w in enumerate(words):
        best = max([lcp(w, u) for j, u in enumerate(words) if j != i], default=0)
        res.append(abbr(w, best + 1))
    return res

w = ["internet", "interval", "intern"]
print("sai:", abbrev_no_class(w), "| đúng:", Solution().wordsAbbreviation(w))

sai: ['internet', 'interval', 'intern'] | đúng: ['i6t', 'i6l', 'i4n']


**Code đúng:** nhóm theo `(len, đầu, cuối)` trước.

### Lỗi 4 — Trả kết quả sai thứ tự

Sau khi nhóm và sắp xếp, phải ghi kết quả vào **đúng chỉ số gốc** (`res[i] = ...`). Nối kết quả theo thứ tự nhóm sẽ làm lệch thứ tự so với `words`.

## 14. Mẹo & nhận dạng pattern

- "Tiền tố ngắn nhất để phân biệt" → **LCP với hàng xóm sau khi sắp xếp** hoặc **Trie có đếm** (giống LC 1166, "shortest unique prefix").
- Luật "nhóm trùng cùng tăng" → thường tương đương công thức đóng `1 + max LCP` — chứng minh bằng "tách rồi không bao giờ trùng lại".
- Chia bài toán thành các **lớp tương đương** (cùng độ dài, cùng ký tự cuối) giúp giảm số cặp phải so.
- Viết mô phỏng đúng luật trước để có **chuẩn đối chiếu**, rồi mới tối ưu.
- Bài liên quan: LC 408 Valid Word Abbreviation, LC 411 Minimum Unique Word Abbreviation, LC 288 Unique Word Abbreviation.

## 15. Biến thể

1. **Shortest unique prefix** (không có phần số/ký tự cuối): `p = 1 + max LCP` với **mọi** từ — Trie đếm.
2. **Viết tắt cả hậu tố** (tăng phần cuối thay vì phần đầu): đảo ngược chuỗi rồi làm tương tự.
3. **Kiểm tra một viết tắt có hợp lệ cho một từ không** (LC 408): duyệt hai con trỏ.
4. **Unique Word Abbreviation (LC 288):** từ điển viết tắt `first + (len-2) + last`.

Demo biến thể 1 và 3:

In [26]:
def shortest_unique_prefix(words):
    root = build_trie(words)
    out = []
    for w in words:
        node = root
        for d, ch in enumerate(w, 1):
            node = node[ch]
            if node["#"] == 1:
                out.append(w[:d]); break
        else:
            out.append(w)
    return out

def valid_abbreviation(word, ab):
    i = j = 0
    while i < len(word) and j < len(ab):
        if ab[j].isdigit():
            if ab[j] == "0":
                return False
            k = 0
            while j < len(ab) and ab[j].isdigit():
                k = k * 10 + int(ab[j]); j += 1
            i += k
        else:
            if word[i] != ab[j]:
                return False
            i += 1; j += 1
    return i == len(word) and j == len(ab)

print(shortest_unique_prefix(["zebra", "dog", "duck", "dove"]))
assert shortest_unique_prefix(["zebra", "dog", "duck", "dove"]) == ["z", "dog", "du", "dov"]
res = Solution().wordsAbbreviation(ex1)
assert all(valid_abbreviation(w, a) for w, a in zip(ex1, res))
print("Mọi viết tắt của ví dụ 1 đều hợp lệ theo LC 408")

['z', 'dog', 'du', 'dov']
Mọi viết tắt của ví dụ 1 đều hợp lệ theo LC 408


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | Lặp: nhóm trùng cùng tăng tiền tố | `O(n L²)` | `O(nL)` | Chuẩn đối chiếu, dễ đúng đề |
| 2. LCP theo cặp | `p = 1 + max LCP` trong lớp, so mọi cặp | `O(n² L)` | `O(n)` | `n` nhỏ, code ngắn |
| 3. Sắp xếp + hàng xóm | LCP lớn nhất ở hàng xóm | `O(n L log n)` | `O(n)` | Lời giải nộp |
| 4. Trie đếm | Nút đầu tiên có count = 1 | `O(n L)` | `O(n L)` | Tối ưu tiệm cận |

## 17. Bài tập

**Bài 1 (đoán output).** `abbr("localization", 1)` và `abbr("localization", 3)` là gì?

**Bài 2 (tính tay).** `words = ["abcde", "abxde", "aycde"]`. Tính lớp, LCP, `p`, và kết quả.

**Bài 3 (điền TODO).** Hoàn thiện `abbr_todo`.

In [27]:
def abbr_todo(w, p):
    # TODO: k = len(w) - p - 1; nếu k <= 1 trả w; ngược lại trả w[:p] + str(k) + w[-1]
    raise NotImplementedError

# Không gọi khi chạy notebook. Tự kiểm tra:
# assert abbr_todo("internal", 1) == "i6l" and abbr_todo("god", 1) == "god"

**Bài 4 (tự viết lại).** Viết lại Cách 4 (Trie) không nhìn code.

**Bài 5 (mini challenge).** Biến thể "tăng hậu tố": viết tắt = ký tự đầu + số + **`q` ký tự cuối**; khi trùng thì cả nhóm tăng `q`. Viết `abbrev_suffix(words)` và kiểm chứng bằng một mô phỏng đúng luật.

In [28]:
def abbrev_suffix_todo(words):
    # TODO
    raise NotImplementedError

# Không gọi khi chạy notebook.

## 18. Gợi ý 3 mức

**Bài 1–2**
- Mức 1: `len("localization") = 12`.
- Mức 2: Bài 1: `"l10n"`, `"loc8n"`.
- Mức 3: Bài 2: cả ba cùng lớp `(5, 'a', 'e')`; sắp: `abcde, abxde, aycde`; LCP kề: 2, 1 → `p = 3, 3, 2` → `"abcde"` (k=1 giữ nguyên), `"abxde"`, `"ay2e"`.

**Bài 3**
- Mức 1: Tính `k` trước.
- Mức 2: `k <= 1` giữ nguyên.
- Mức 3: Ghép chuỗi.

**Bài 4**
- Mức 1: Nhóm theo lớp, dựng trie mỗi lớp.
- Mức 2: Mỗi nút lưu `count`.
- Mức 3: Dừng ở nút đầu tiên có `count == 1`.

**Bài 5**
- Mức 1: Đảo ngược mỗi từ: "tăng hậu tố" thành "tăng tiền tố" của chuỗi đảo — nhưng ký tự **đầu** gốc thành ký tự **cuối** của chuỗi đảo.
- Mức 2: Lớp vẫn là `(len, đầu, cuối)`.
- Mức 3: Với chuỗi đảo `r`, viết tắt gốc = `w[0] + str(k) + w[-q:]` với `q = 1 + max LCP` của các chuỗi đảo trong lớp.

## 19. Đáp án tham khảo

In [29]:
# Bài 1, 2
assert abbr("localization", 1) == "l10n" and abbr("localization", 3) == "loc8n"
assert Solution().wordsAbbreviation(["abcde", "abxde", "aycde"]) == ["abcde", "abxde", "ay2e"]

# Bài 3
def abbr_sol(w, p):
    k = len(w) - p - 1
    return w if k <= 1 else w[:p] + str(k) + w[-1]
assert abbr_sol("internal", 1) == "i6l" and abbr_sol("god", 1) == "god"

# Bài 4
def abbrev_trie_ex(words):
    groups = defaultdict(list)
    for i, w in enumerate(words):
        groups[(len(w), w[0], w[-1])].append(i)
    res = [""] * len(words)
    for idxs in groups.values():
        root = {}
        for i in idxs:
            node = root
            for ch in words[i]:
                node = node.setdefault(ch, {"#": 0}); node["#"] += 1
        for i in idxs:
            node, p = root, len(words[i])
            for d, ch in enumerate(words[i], 1):
                node = node[ch]
                if node["#"] == 1:
                    p = d; break
            res[i] = abbr_sol(words[i], p)
    return res

random.seed(61)
for _ in range(300):
    ws = list({"".join(random.choice("ab") for _ in range(random.randint(2, 7))) for _ in range(random.randint(1, 10))})
    assert abbrev_trie_ex(ws) == abbrev_simulate(ws)
print("Bài 1–4 OK")

Bài 1–4 OK


In [30]:
# Bài 5: tăng hậu tố
def abbr_suf(w, q):
    k = len(w) - q - 1
    return w if k <= 1 else w[0] + str(k) + w[-q:]

def abbrev_suffix_simulate(words):
    n = len(words); q = [1] * n
    ab = [abbr_suf(w, 1) for w in words]
    while True:
        cnt = Counter(ab)
        dup = [i for i in range(n) if cnt[ab[i]] > 1]
        if not dup:
            return ab
        for i in dup:
            q[i] += 1; ab[i] = abbr_suf(words[i], q[i])

def abbrev_suffix(words):
    groups = defaultdict(list)
    for i, w in enumerate(words):
        groups[(len(w), w[0], w[-1])].append(i)
    res = [""] * len(words)
    for idxs in groups.values():
        rev = sorted(idxs, key=lambda i: words[i][::-1])
        m = len(rev)
        adj = [lcp(words[rev[k]][::-1], words[rev[k + 1]][::-1]) for k in range(m - 1)]
        for k, i in enumerate(rev):
            best = max(adj[k - 1] if k > 0 else 0, adj[k] if k < m - 1 else 0)
            res[i] = abbr_suf(words[i], best + 1)
    return res

print(abbrev_suffix(["internal", "interval", "intension", "intrusion"]))
random.seed(62)
for _ in range(400):
    ws = list({"".join(random.choice("abc") for _ in range(random.randint(2, 8))) for _ in range(random.randint(1, 10))})
    assert abbrev_suffix(ws) == abbrev_suffix_simulate(ws), ws
print("Bài 5 OK")

['i4nal', 'i4val', 'i3nsion', 'i3usion']
Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template:**

```python
groups = defaultdict(list)
for i, w in enumerate(words): groups[(len(w), w[0], w[-1])].append(i)
res = [""] * len(words)
for idxs in groups.values():
    idxs.sort(key=lambda i: words[i])
    adj = [lcp(words[idxs[k]], words[idxs[k+1]]) for k in range(len(idxs) - 1)]
    for k, i in enumerate(idxs):
        best = max(adj[k-1] if k else 0, adj[k] if k < len(adj) else 0)
        res[i] = abbr(words[i], best + 1)      # abbr: k<=1 thì giữ nguyên
return res
```

**Checklist trước khi submit:**

- [ ] `abbr`: `k = len - p - 1`, `k <= 1` → giữ nguyên.
- [ ] Nhóm theo `(độ dài, ký tự đầu, ký tự cuối)`.
- [ ] `p = 1 + LCP lớn nhất` (hàng xóm sau khi sắp xếp hoặc trie).
- [ ] Ghi kết quả đúng chỉ số gốc.
- [ ] Test `["aa","aaa"]`, từ dài 2–3, từ có tiền tố chung rất dài.
- [ ] Đối chiếu với mô phỏng đúng luật trên input ngẫu nhiên.